# Notebook 08 — Export & Validate All Models → manifest.json

Verifies every required model exists, validates they load correctly, generates SHA-256 checksums, and writes `manifest.json` to the shared volume.

In [ ]:
import os, hashlib, json, shutil
from datetime import datetime, timezone
from pathlib import Path

EXPORTS_DIR = Path(os.getenv('EXPORTS_DIR', r'C:\Users\MJ\Desktop\Agric\jupyter\exports\models'))
PROD_DIR = Path(os.getenv('PROD_MODELS_DIR', r'C:\Users\MJ\Desktop\Agric\shared_volumes\models'))
COPY_TO_PROD = os.getenv('COPY_TO_PROD', 'false').lower() == 'true'
MODEL_VERSION = os.getenv('MODEL_VERSION', 'v1')

REQUIRED = [
    'crop_classifier_v1.onnx',
    'disease_detector_v1.onnx',
    'quality_grader_v1.onnx',
    'price_predictor_v1.pkl',
    'price_scaler_v1.pkl',
    'risk_scorer_v1.pkl',
    'risk_scaler_v1.pkl',
    'fraud_detector_v1.pkl',
    'fraud_scaler_v1.pkl',
    'demand_forecaster_v1.pkl',
    'demand_scaler_v1.pkl',
]

METADATA_FILES = {
    'crop_classifier_v1.onnx': 'crop_classifier_metadata.json',
    'disease_detector_v1.onnx': 'disease_detector_metadata.json',
    'quality_grader_v1.onnx': 'quality_grader_metadata.json',
    'price_predictor_v1.pkl': 'price_predictor_metadata.json',
    'risk_scorer_v1.pkl': 'risk_scorer_metadata.json',
    'fraud_detector_v1.pkl': 'fraud_detector_metadata.json',
    'demand_forecaster_v1.pkl': 'demand_forecaster_metadata.json',
}

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

missing = [f for f in REQUIRED if not (EXPORTS_DIR / f).exists()]
missing_meta = [m for m in METADATA_FILES.values() if not (EXPORTS_DIR / m).exists()]
if missing:
    raise FileNotFoundError(f'Missing model files. Run training notebooks first: {missing}')
if missing_meta:
    raise FileNotFoundError(f'Missing metadata files. Re-run relevant notebooks: {missing_meta}')
print('All required model and metadata files present.')
print('EXPORTS_DIR:', EXPORTS_DIR)
print('PROD_DIR:', PROD_DIR)

In [ ]:
import onnxruntime as ort
import numpy as np

def test_onnx(path, input_shape=(1, 3, 224, 224)):
    sess = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    dummy = np.random.randn(*input_shape).astype(np.float32)
    out = sess.run(None, {sess.get_inputs()[0].name: dummy})
    print(f'  ✓ {path.name} — output shape {out[0].shape}')
    return True

print('ONNX model validation:')
for onnx_name in ['crop_classifier_v1.onnx', 'disease_detector_v1.onnx', 'quality_grader_v1.onnx']:
    test_onnx(EXPORTS_DIR / onnx_name)

In [ ]:
import joblib

print('sklearn .pkl model validation:')
for pkl_name in ['price_predictor_v1.pkl', 'price_scaler_v1.pkl', 'risk_scorer_v1.pkl', 'risk_scaler_v1.pkl', 'fraud_detector_v1.pkl', 'fraud_scaler_v1.pkl', 'demand_forecaster_v1.pkl', 'demand_scaler_v1.pkl']:
    p = EXPORTS_DIR / pkl_name
    m = joblib.load(p)
    print(f'  ✓ {pkl_name} — type: {type(m).__name__}')

In [ ]:
metadata_by_model = {}
for model_file, metadata_file in METADATA_FILES.items():
    metadata_path = EXPORTS_DIR / metadata_file
    metadata = json.loads(metadata_path.read_text())
    metadata_by_model[model_file] = metadata
    if metadata.get('sha256') and metadata['sha256'] != sha256_file(EXPORTS_DIR / model_file):
        raise ValueError(f'Metadata checksum mismatch for {model_file}')

manifest = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'sovereign_status': 'PRODUCTION_READY',
    'model_version': MODEL_VERSION,
    'exports_dir': str(EXPORTS_DIR),
    'models': {},
}

for fname in REQUIRED:
    fpath = EXPORTS_DIR / fname
    checksum = sha256_file(fpath)
    size_kb = fpath.stat().st_size // 1024
    manifest['models'][fname] = {
        'sha256': checksum,
        'size_kb': size_kb,
        'status': 'ready',
    }
    if fname in metadata_by_model:
        manifest['models'][fname]['metadata'] = metadata_by_model[fname]

manifest_path = EXPORTS_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
print('manifest.json written:')
print(json.dumps(manifest, indent=2))

In [ ]:
all_ready = all(v.get('status') == 'ready' for v in manifest['models'].values())
if not all_ready:
    missing_m = [k for k, v in manifest['models'].items() if v.get('status') != 'ready']
    raise RuntimeError(f'{len(missing_m)} model(s) still missing: {missing_m}')

if COPY_TO_PROD:
    PROD_DIR.mkdir(parents=True, exist_ok=True)
    for fname in REQUIRED + ['manifest.json'] + list(METADATA_FILES.values()):
        shutil.copy2(EXPORTS_DIR / fname, PROD_DIR / fname)
    print('Copied validated model bundle to production directory:', PROD_DIR)
else:
    print('COPY_TO_PROD=false, leaving validated bundle in exports directory only.')

print('\n✅ ALL MODELS READY FOR PRODUCTION')
print('Validated export bundle:', EXPORTS_DIR)
print('Backend container loads from /app/models or MODELS_PATH when the bundle is copied/mounted there.')